In [11]:
!pip install -q contractions
!pip install -q spacy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.8/114.8 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 84.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [12]:
import pandas as pd
import re
import nltk
import spacy
import contractions

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [3]:
# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [13]:
nlp = spacy.load("en_core_web_sm")

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [14]:
df = pd.read_csv("news_dataset.csv")

print("Dataset Shape:", df.shape)

display(df.head())

Dataset Shape: (6000, 5)


,title,text,subject,date,label
0,WATCH: Trump Just Told All The Anti-Gay Bigot...,A whole lot of evangelical Trump voters just d...,News,"November 13, 2016",0
1,"China backs U.N. call for justice in Yemen, U....",GENEVA (Reuters) - China signaled on Wednesday...,worldnews,"September 13, 2017",1
2,THE PEOPLE’S PRESIDENT: Trump Meets With Coal ...,,politics,"Feb 16, 2017",0
3,WSJ REPORTER RIPS INTO DEM CANDIDATES For Thei...,THE WSJ S MARY KISSEL NAILS IT ON THE DEM DEBA...,politics,"Nov 15, 2015",0
4,Lawmakers aim to delay U.S. ceding control of ...,WASHINGTON (Reuters) - Critics of a plan for t...,politicsNews,"September 13, 2016",1


In [15]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDataset Information:")
df.info()

Columns:
['title', 'text', 'subject', 'date', 'label']

Missing Values:
title      0
text       0
subject    0
date       0
label      0
dtype: int64

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    6000 non-null   object
 1   text     6000 non-null   object
 2   subject  6000 non-null   object
 3   date     6000 non-null   object
 4   label    6000 non-null   int64 
dtypes: int64(1), object(4)
memory usage: 234.5+ KB


In [16]:
df['text'] = df['text'].fillna('')

# Keep original text for comparison
df['original_text'] = df['text']

In [17]:
stop_words = set(stopwords.words('english'))

print("Total stopwords:", len(stop_words))
print(list(stop_words)[:20])

Total stopwords: 198
["wasn't", 'i', 'of', 'its', "he'd", 'she', 'this', 'having', 'haven', 'be', 'again', 'hers', "hadn't", 'after', 'so', "i'm", "they'll", 'wasn', 'during', "wouldn't"]


In [18]:
def preprocess_text(text):

    # Convert to string
    text = str(text)

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Expand contractions using contractions library
    text = contractions.fix(text)

    # 3. Remove HTML tags using RegEx
    text = re.sub(r'<.*?>', ' ', text)

    # 4. Remove URLs using RegEx
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # 5. Remove email addresses using RegEx
    text = re.sub(r'\S+@\S+', ' ', text)

    # 6. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # 7. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # 8. Tokenization using NLTK
    tokens = word_tokenize(text)

    # 9. Stop-word removal using NLTK
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    # 10. Lemmatization using spaCy
    doc = nlp(" ".join(tokens))

    tokens = [
        token.lemma_
        for token in doc
        if not token.is_space
    ]

    # 11. Convert tokens back to text
    return " ".join(tokens)

In [19]:
sample = "I don't think this news is accurate. They're reporting it incorrectly."

print("Before:")
print(sample)

print("\nAfter using Contractions library:")
print(contractions.fix(sample))

Before:
I don't think this news is accurate. They're reporting it incorrectly.

After using Contractions library:
I do not think this news is accurate. They are reporting it incorrectly.


In [20]:
sample = df.loc[0, 'text']

print("ORIGINAL TEXT:")
print(sample[:1000])

print("\n" + "="*80)

print("PROCESSED TEXT:")
print(preprocess_text(sample)[:1000])

ORIGINAL TEXT:
A whole lot of evangelical Trump voters just discovered they got duped.By picking Mike Pence as his vice-president, Donald Trump sent a message to the LGBT community that their rights are in jeopardy of being rolled back.In fact, one of the reasons why so many conservative  Christians  tossed their alleged  morality  out the window to vote for him was precisely because they believed he would pass a constitutional amendment banning same-sex marriage or pack the courts with anti-gay bigots to strike down all the rulings in favor of marriage equality.But during his interview on 60 minutes with Leslie Stahl on Sunday night, Trump once again backed off another campaign promise. Do you support marriage equality?  Stahl asked Trump. It s irrelevant because it s already settled,  Trump replied.  It s law. It was settled in the Supreme Court. I mean, it s done. When Stahl pointed out that Trump s Supreme Court nominees could reverse Obergefell v. Hodges, the case that made same-s

In [21]:
print("Starting preprocessing...")

df['processed_text'] = df['text'].apply(preprocess_text)

print("Preprocessing completed!")

Starting preprocessing...
Preprocessing completed!


In [22]:
pd.set_option('display.max_colwidth', 500)

display(
    df[['original_text', 'processed_text']].head(5)
)

,original_text,processed_text
0,"A whole lot of evangelical Trump voters just discovered they got duped.By picking Mike Pence as his vice-president, Donald Trump sent a message to the LGBT community that their rights are in jeopardy of being rolled back.In fact, one of the reasons why so many conservative Christians tossed their alleged morality out the window to vote for him was precisely because they believed he would pass a constitutional amendment banning same-sex marriage or pack the courts with anti-gay bigots to ...",whole lot evangelical trump voter discover get dupe pick mike pence vice president donald trump send message lgbt community right jeopardy roll back fact one reason many conservative christian toss allege morality window vote precisely believe would pass constitutional amendment ban sex marriage pack court anti gay bigot strike ruling favor marriage equality interview minute leslie stahl sunday night trump back another campaign promise support marriage equality stahl ask trump irrelevant alr...
1,"GENEVA (Reuters) - China signaled on Wednesday it was willing to back an international inquiry into atrocities in Yemen, as demanded by the U.N. High Commissioner for Human Rights, but Saudi Arabia and the United States said they did not support the idea. For three years running U.N. human rights chief Zeid Ra ad al-Hussein has asked the 47 countries in the U.N. Human Rights Council to set up an independent investigation into Yemen s war, which has killed at least 10,000, destroyed the econo...",geneva reuters china signal wednesday willing back international inquiry atrocity yemen demand n high commissioner human right saudi arabia united states say support idea three year run n human right chief zeid ra ad al hussein ask country n human rights council set independent investigation yemen war kill least destroy economy lead cholera epidemic push millions brink famine despite plea twice support saudi plan let yemen investigate wednesday netherlands canada unveil draft resolution esta...
2,,
3,"THE WSJ S MARY KISSEL NAILS IT ON THE DEM DEBATE WITH THE THREE CANDIDATES WHO WERE GIVING NO SOLUTIONS ON ISIS: It was one of the most remarkable displays of unintelligible garbage rhetoric that I have ever seen. Amen to that!Mary Kissel: The barbarians are at the gates. We ve seen attacks now from London to Madrid to Beirut to over the Sinai. It is time for American leadership. Hillary said this isn t America s fight. Look, Democrats are dangerously divorced from reality. Hillary says th...",wsj mary kissel nail dem debate three candidate give solution isis one remarkable display unintelligible garbage rhetoric ever see amen mary kissel barbarians gates see attack london madrid beirut sinai time american leadership hillary say america fight look democrat dangerously divorce reality hillary say president obama want close gitmo say contain people contain bernie sander lead poll say great challenge climate change need american leadership majority american people understand need sen...
4,"WASHINGTON (Reuters) - Critics of a plan for the U.S. government to cede control of the Internet’s technical management to other countries might succeed in delaying such a move in Congress, a senior lawmaker said on Tuesday. Senator John Thune, a senior Republican from South Dakota, told reporters lawmakers are “trying to work out ... what would be effective in terms of slowing this.” The U.S. Commerce Department oversees the Internet’s management largely because it was invented in the Unite...",washington reuters critic plan government cede control internet technical management country might succeed delay move congress senior lawmaker say tuesday senator john thune senior republican south dakota tell reporter lawmaker try work would effective term slow commerce department oversee internet management largely invent united states republican politician include senator ted cruz texas want block handover global stakeholder business tech expert pub

In [25]:
df.to_csv(
    "news_dataset_preprocessed.csv",
    index=False
)

print("Preprocessed dataset saved successfully!")

Preprocessed dataset saved successfully!


In [26]:
from google.colab import files

files.download("news_dataset_preprocessed.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>